In [2]:
import os
import json

from zipfile import ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [3]:
data = pd.read_csv('IMDB Dataset.csv')

In [4]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
data.shape

(50000, 2)

In [6]:
data['sentiment'].value_counts()

sentiment
positive    25000
negative    25000
Name: count, dtype: int64

In [7]:
data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace = True)

C:\Users\Admin\AppData\Local\Temp\ipykernel_9168\2164277587.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment": {"positive": 1, "negative": 0}}, inplace = True)


In [8]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [9]:
data['sentiment'].value_counts()

sentiment
1    25000
0    25000
Name: count, dtype: int64

In [10]:
train_data, test_data = train_test_split(data, test_size = 0.2, random_state = 42)

In [11]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


### Data Preprocessing

In [12]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(train_data['review'])
x_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]), maxlen=200)
x_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]), maxlen=200)

In [13]:
print(x_train)

[[1935    1 1200 ...  205  351 3856]
 [   3 1651  595 ...   89  103    9]
 [   0    0    0 ...    2  710   62]
 ...
 [   0    0    0 ... 1641    2  603]
 [   0    0    0 ...  245  103  125]
 [   0    0    0 ...   70   73 2062]]


In [14]:
print(x_test)

[[   0    0    0 ...  995  719  155]
 [  12  162   59 ...  380    7    7]
 [   0    0    0 ...   50 1088   96]
 ...
 [   0    0    0 ...  125  200 3241]
 [   0    0    0 ... 1066    1 2305]
 [   0    0    0 ...    1  332   27]]


In [15]:
y_train = train_data["sentiment"]
y_test = test_data["sentiment"]

In [16]:
print(y_train)

39087    0
30893    0
45278    1
16398    0
13653    0
        ..
11284    1
44732    1
38158    0
860      1
15795    1
Name: sentiment, Length: 40000, dtype: int64


### LSTM - Long Short Term Memory

In [17]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=200))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

C:\Users\Admin\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [19]:
model.compile(optimizer='Adam', loss='binary_crossentropy', metrics=['accuracy'])

### Train the model

In [20]:
model.fit(x_train, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 174s 333ms/step - accuracy: 0.7894 - loss: 0.4575 - val_accuracy: 0.8395 - val_loss: 0.3776
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 240s 481ms/step - accuracy: 0.8635 - loss: 0.3351 - val_accuracy: 0.8485 - val_loss: 0.3540
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 239s 477ms/step - accuracy: 0.8786 - loss: 0.2992 - val_accuracy: 0.8699 - val_loss: 0.3231
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 241s 482ms/step - accuracy: 0.8948 - loss: 0.2609 - val_accuracy: 0.8715 - val_loss: 0.3251
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 248s 496ms/step - accuracy: 0.9063 - loss: 0.2361 - val_accuracy: 0.8780 - val_loss: 0.3321


In [21]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - accuracy: 0.8841 - loss: 0.3139
Test Loss: 0.31393492221832275
Test Accuracy: 0.8841000199317932


### Buid a predictive sysytem

In [31]:
def predict_sentiment(review):
    sequence = tokenizer.texts_to_sequences([review])
    padded_sequence = pad_sequences(sequence, maxlen=200)
    prediction = model.predict(padded_sequence)
    sentiment = "positive" if prediction[0][0] > 0.5 else "negative"
    return sentiment

In [32]:
new_review = "this movie is fantastic. I love it"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of this review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step
The sentiment of this review is: positive


In [35]:
new_review = "this movie is not good. i don't like this movie"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of this review is: {sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 103ms/step
The sentiment of this review is: negative
